In [ ]:
!pip install pandas scikit-learn mlflow joblib

In [ ]:
import urllib.request
import pandas as pd
from sklearn.model_selection import train_test_split

urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/ops4life/mlops-get-started/main/employee_attrition.csv",
    "employee_attrition.csv"
)

# Run prior pipeline steps to produce train.csv
df_raw = pd.read_csv("employee_attrition.csv")

df_fe = df_raw.copy()
df_fe['Attrition'] = df_fe['Attrition'].map({'Stayed': 0, 'Left': 1}).astype('int')
binary_cols = ["Remote Work", "Leadership Opportunities", "Innovation Opportunities", "Overtime"]
for col in binary_cols:
    if col in df_fe.columns:
        df_fe[col] = df_fe[col].map({"Yes": 1, "No": 0}).astype("Int64")
ordinal_maps = {
    "Work-Life Balance": {"Poor": 1, "Fair": 2, "Good": 3, "Excellent": 4},
    "Job Satisfaction": {"Low": 1, "Medium": 2, "High": 3, "Very High": 4},
    "Performance Rating": {"Low": 1, "Below Average": 2, "Average": 3, "High": 4},
    "Employee Recognition": {"Low": 1, "Medium": 2, "High": 3, "Very High": 4},
    "Company Reputation": {"Poor": 1, "Fair": 2, "Good": 3, "Excellent": 4},
    "Job Level": {"Entry": 1, "Mid": 2, "Senior": 3},
    "Company Size": {"Small": 1, "Medium": 2, "Large": 3},
    "Education Level": {"High School": 1, "Bachelor's Degree": 2, "Master's Degree": 3, "Associate Degree": 4, "PhD": 5},
}
for col, mapping in ordinal_maps.items():
    if col in df_fe.columns:
        df_fe[col] = df_fe[col].map(mapping)
df_fe["OverallSatisfaction"] = ((df_fe["Work-Life Balance"] + df_fe["Job Satisfaction"] + df_fe["Employee Recognition"]) / 3).round().astype("Int64")
df_fe = df_fe.drop(columns=["Work-Life Balance", "Job Satisfaction", "Employee Recognition"])
df_fe['Opportunities'] = df_fe['Leadership Opportunities'] + df_fe['Innovation Opportunities']
df_fe = df_fe.drop(columns=['Leadership Opportunities', 'Innovation Opportunities'])
df_fe["AnnualIncome"] = pd.cut(df_fe["Monthly Income"] * 12, bins=[0, 240000, 420000, 600000, 2000000, float("inf")], labels=[0, 1, 2, 3, 4], include_lowest=True).astype("Int64")
df_fe = df_fe.drop(columns=["Monthly Income"])
df_fe['AgeGroup'] = pd.cut(df_fe['Age'], bins=[17, 25, 35, 45, 60, 65], labels=[1, 2, 3, 4, 5]).astype("Int64")
df_fe = df_fe.drop(columns=['Age'])
df_fe["Years at Company"] = (df_fe["Years at Company"] / 12).round(2)
df_fe["Company Tenure"] = (df_fe["Company Tenure"] / 12).round(2)
df_fe['RoleStagnationRatio'] = (df_fe['Years at Company'] / (df_fe['Company Tenure'] + 1)).round(3)
df_fe['TenureGap'] = df_fe['Company Tenure'] - df_fe['Years at Company']
df_fe['EarlyCompanyTenureRisk'] = (df_fe['Years at Company'] <= 2).astype("Int64")
df_fe["LongTenureLowRoleRisk"] = ((df_fe["Company Tenure"] > 5) & (df_fe["Job Level"] <= 2)).astype("Int64")
drop_cols = [c for c in ['Employee ID', 'Job Role', 'Distance from Home', 'Marital Status', 'Gender', 'dataset_type'] if c in df_fe.columns]
df_fe = df_fe.drop(columns=drop_cols)

X = df_fe.drop(columns=['Attrition'])
y = df_fe['Attrition']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train = X_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
train_df = pd.concat([X_train, y_train], axis=1)
train_df.to_csv("train.csv", index=False)
print("Setup complete: train.csv ready.")

In [ ]:
import pandas as pd
import joblib
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Columns that need scaling
NUMERIC_COLS = ['Years at Company', 'Company Tenure', 'RoleStagnationRatio', 'TenureGap']


def train_data(df):
    X_train = df.drop(columns=['Attrition'])
    y_train = df['Attrition']

    # Get integer indices for numeric columns
    numeric_indices = [X_train.columns.get_loc(col) for col in NUMERIC_COLS]

    preprocessor = ColumnTransformer(
        transformers=[('scaler', StandardScaler(), numeric_indices)],
        remainder='passthrough'
    )

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced'))
    ])

    mlflow.set_tracking_uri("file:./mlruns")
    mlflow.set_experiment("employee-attrition")

    with mlflow.start_run():
        mlflow.log_param("model", "LogisticRegression")
        mlflow.log_param("class_weight", "balanced")

        print("Training the model....")
        pipeline.fit(X_train, y_train)
        print("Training completed...")

        mlflow.sklearn.log_model(pipeline, artifact_path="model")
        joblib.dump(pipeline, "model.pkl")

    return pipeline


df_train = pd.read_csv("train.csv")
train_data(df_train)